## 自建回测框架测试

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==================== 数据准备 ====================
df = pd.read_csv('test_data_2.csv', parse_dates=['date'])
df = df.sort_values(['date', 'code'])

# 透视数据：日期为索引，股票为列
prices = df.pivot(index='date', columns='code', values='pre_close')
bvps = df.pivot(index='date', columns='code', values='bvps')

# 向前填充 BVPS（BVPS 不每日变化）
bvps = bvps.ffill()

# 删除缺失值
prices = prices.dropna()
bvps = bvps.dropna()

print("Data shape:", prices.shape)
print("Date range:", prices.index.min(), "to", prices.index.max())
print("Stocks:", list(prices.columns))

# ==================== 选股逻辑 ====================
def select_stock(bvps_series, date):
    """返回 BVPS 最大的股票代码"""
    return bvps_series.loc[date].idxmax()

# 获取所有日期
all_dates = prices.index

# 确定每周的第一个交易日作为调仓日
weekly_dates = []
current_week = None
current_year = None
for date in all_dates:
    year, week = date.isocalendar()[:2]
    if (year, week) != (current_year, current_week):
        weekly_dates.append(date)
        current_year, current_week = year, week

print(f"\nRebalancing on {len(weekly_dates)} dates")

# ==================== 回测框架 ====================
init_cash = 100000.0
cash = init_cash
holding_stock = None      # 当前持有的股票代码
holding_shares = 0.0      # 当前持有的股数

# 记录每日组合价值
equity_curve = pd.Series(index=all_dates, dtype=float)

# 交易记录
trades = []   # 每个元素为 (buy_date, sell_date, buy_price, sell_price, return, duration)

# 辅助函数：计算收益率
def calc_return(buy_price, sell_price):
    return (sell_price - buy_price) / buy_price

# 回测主循环
for i, date in enumerate(all_dates):
    price_today = prices.loc[date]        # 当日所有股票价格
    is_rebalance = date in weekly_dates

    # 如果是调仓日，先卖出当前持仓（如果有），然后买入新股票
    if is_rebalance:
        # 卖出当前持仓（如果有）
        if holding_stock is not None:
            sell_price = price_today[holding_stock]
            cash += holding_shares * sell_price
            # 记录交易完成
            trades[-1]['sell_date'] = date
            trades[-1]['sell_price'] = sell_price
            trades[-1]['return'] = calc_return(trades[-1]['buy_price'], sell_price)
            trades[-1]['duration'] = (date - trades[-1]['buy_date']).days
            holding_stock = None
            holding_shares = 0.0

        # 选择新股票
        new_stock = select_stock(bvps, date)
        buy_price = price_today[new_stock]
        # 全仓买入
        holding_shares = cash / buy_price
        cash = 0.0
        holding_stock = new_stock
        # 记录新交易开始
        trades.append({
            'buy_date': date,
            'buy_price': buy_price,
            'stock': new_stock
        })

    # 计算当日组合价值（收盘时）
    if holding_stock is not None:
        value = holding_shares * price_today[holding_stock]
    else:
        value = cash  # 现金，没有持仓（可能是初始状态或刚卖出未买入？实际在调仓日买入后不会为现金）
    equity_curve[date] = value

# 回测结束：若仍有持仓，以最后一天价格平仓
if holding_stock is not None:
    last_date = all_dates[-1]
    last_price = prices.loc[last_date, holding_stock]
    cash = holding_shares * last_price
    # 记录最后一笔交易
    trades[-1]['sell_date'] = last_date
    trades[-1]['sell_price'] = last_price
    trades[-1]['return'] = calc_return(trades[-1]['buy_price'], last_price)
    trades[-1]['duration'] = (last_date - trades[-1]['buy_date']).days
    # 最后价值已包含在 equity_curve 中（当日价值是用持仓计算的，但最后一天价值已记录）
    # 但为了准确，最后一天价值应该是现金（因为平仓了），我们重新计算最后一天价值
    equity_curve[last_date] = cash  # 平仓后只有现金

# 将交易记录转为 DataFrame
trades_df = pd.DataFrame(trades)

# ==================== 统计指标计算 ====================
# 每日收益率
daily_returns = equity_curve.pct_change().dropna()
total_return = (equity_curve.iloc[-1] / init_cash) - 1

# 年化收益率（假设 252 个交易日）
num_days = len(equity_curve)
annualized_return = (1 + total_return) ** (252 / num_days) - 1 if num_days > 0 else 0

# 夏普比率（无风险利率设为 0）
sharpe_ratio = (annualized_return) / (daily_returns.std() * np.sqrt(252)) if daily_returns.std() != 0 else 0

# 最大回撤
cumulative = (1 + daily_returns).cumprod()
running_max = cumulative.cummax()
drawdown = (cumulative - running_max) / running_max
max_drawdown = drawdown.min()

# 交易统计
if len(trades_df) > 0:
    returns = trades_df['return'].values
    durations = trades_df['duration'].values
    num_trades = len(returns)
    avg_return = np.mean(returns)
    median_return = np.median(returns)
    best_return = np.max(returns)
    worst_return = np.min(returns)
    avg_duration = np.mean(durations)

    winning = returns[returns > 0]
    losing = returns[returns <= 0]
    win_rate = len(winning) / num_trades if num_trades > 0 else 0
    avg_win = np.mean(winning) if len(winning) > 0 else 0
    avg_loss = np.mean(losing) if len(losing) > 0 else 0
else:
    num_trades = 0
    avg_return = median_return = best_return = worst_return = avg_duration = 0
    win_rate = avg_win = avg_loss = 0

# ==================== 基准比较（等权重） ====================
# 构造等权重组合每日价值
n_stocks = len(prices.columns)
equal_weight_portfolio = pd.Series(index=prices.index, dtype=float)
for date in prices.index:
    equal_weight_portfolio[date] = (prices.loc[date].mean() / prices.iloc[0].mean()) * init_cash
# 简化：每日价值 = 初始现金 * (当日平均价格 / 初始平均价格)
equal_weight_returns = equal_weight_portfolio.pct_change().dropna()
equal_weight_total_return = (equal_weight_portfolio.iloc[-1] / init_cash) - 1

excess_return = total_return - equal_weight_total_return

# ==================== 打印结果 ====================
print("\n" + "="*60)
print("BACKTEST RESULTS")
print("="*60)
print(f"\nInitial Capital: ${init_cash:,.2f}")
print(f"Final Value: ${equity_curve.iloc[-1]:,.2f}")
print(f"Total Return: {total_return:.2%}")
print(f"Total Profit: ${equity_curve.iloc[-1] - init_cash:,.2f}")
print(f"Annualized Return: {annualized_return:.2%}")
print(f"Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")

print("\n" + "="*60)
print("TRADE STATISTICS")
print("="*60)
if num_trades > 0:
    print(f"Number of Trades: {num_trades}")
    print(f"Average Trade Return: {avg_return:.2%}")
    print(f"Median Trade Return: {median_return:.2%}")
    print(f"Best Trade: {best_return:.2%}")
    print(f"Worst Trade: {worst_return:.2%}")
    print(f"Average Holding Period: {avg_duration:.0f} days")
    print(f"\nWinning Trades: {len(winning)} ({win_rate:.1%})")
    print(f"  Average Win: {avg_win:.2%}")
    if len(losing) > 0:
        print(f"Losing Trades: {len(losing)} ({1-win_rate:.1%})")
        print(f"  Average Loss: {avg_loss:.2%}")
else:
    print("No trades executed")

print("\n" + "="*60)
print("BENCHMARK COMPARISON")
print("="*60)
print(f"Strategy Return: {total_return:.2%}")
print(f"Equal Weight Benchmark Return: {equal_weight_total_return:.2%}")
print(f"Excess Return: {excess_return:.2%}")

# ==================== 绘图 ====================
# 计算累计收益率曲线
cumulative_returns = equity_curve / init_cash - 1

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=('Equity Curve', 'Cumulative Returns'),
    vertical_spacing=0.15,
    row_heights=[0.5, 0.5]
)

fig.add_trace(
    go.Scatter(x=equity_curve.index, y=equity_curve, mode='lines', name='Portfolio Value',
               line=dict(color='blue', width=2), fill='tozeroy', fillcolor='rgba(0,100,200,0.2)'),
    row=1, col=1
)
fig.add_hline(y=init_cash, line_dash="dash", line_color="gray", row=1, col=1)

fig.add_trace(
    go.Scatter(x=cumulative_returns.index, y=cumulative_returns, mode='lines', name='Cumulative Returns',
               line=dict(color='green', width=2), fill='tozeroy', fillcolor='rgba(0,200,0,0.2)'),
    row=2, col=1
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", row=2, col=1)

fig.update_layout(
    title="Strategy Performance: Weekly Selection of Highest BVPS Stock",
    height=800, showlegend=True, hovermode='x unified'
)
fig.update_yaxes(title_text="Portfolio Value ($)", row=1, col=1)
fig.update_yaxes(title_text="Cumulative Returns (%)", row=2, col=1, tickformat='.0%')
fig.update_xaxes(title_text="Date", row=2, col=1)
fig.show()

C:\Users\Administrator\AppData\Roaming\Python\Python311\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Data shape: (22, 3)
Date range: 2020-01-02 00:00:00 to 2020-01-31 00:00:00
Stocks: ['000001.SZ', '000002.SZ', '600036.SH']

Rebalancing on 5 dates

BACKTEST RESULTS

Initial Capital: $100,000.00
Final Value: $108,642.05
Total Return: 8.64%
Total Profit: $8,642.05
Annualized Return: 158.43%
Sharpe Ratio: 8.42
Max Drawdown: -7.08%

TRADE STATISTICS
Number of Trades: 5
Average Trade Return: 1.75%
Median Trade Return: 2.31%
Best Trade: 6.61%
Worst Trade: -2.70%
Average Holding Period: 6 days

Winning Trades: 3 (60.0%)
  Average Win: 4.71%
Losing Trades: 2 (40.0%)
  Average Loss: -2.70%

BENCHMARK COMPARISON
Strategy Return: 8.64%
Equal Weight Benchmark Return: 7.96%
Excess Return: 0.68%
